## Preprocesamiento con Pyspark
A continuación, se procederá a replicar el flujo realizado con Scikit learn, pero utilizando Pyspark. No se repetirá el procedimiento d selección de variables, las columnas resultantes del proceso con Mutual Information serán utlizadas directamente en esta sección.

Primero, se configurará el entorno, para eso se definirá una variable que contendrá el directorio de descarga de Java 17, la versión compatible con Pyspark 4.1.1. Luego, se creará la sesión con Spark Session, el cuál podría entenderse como el ambiente desde el que se leerán los datos y se realizarán las transformaciones.

In [3]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.18/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: /opt/homebrew/Cellar/openjdk@17/17.0.18/libexec/openjdk.jdk/Contents/Home


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
         .appName("LendingClub_Preprocessing")
         .master("local[2]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.enabled", "false")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession activa — versión Spark: {spark.version}")

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/19 09:05:59 WARN Utils: Your hostname, Tiffanys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.29 instead (on interface en0)
26/03/19 09:05:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 09:06:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession activa — versión Spark: 4.1.1


### Lectura de datos y filtrado inicial
Ahora, se comenzará leyendo los datos, pero filtrando inmediatamente por las columnas encontradas en el mutual information para ahorrar tiempo en el procesado de datos.

In [5]:
CSV_PATH = "/Users/pctm/Documents/archive/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

SELECTED_FEATURES = [
    'int_rate', 'dti', 'num_actv_rev_tl', 'fico_range_high', 'grade',
    'acc_open_past_24mths', 'funded_amnt', 'term', 'mort_acc', 'loan_amnt',
    'revol_util', 'fico_range_low', 'inq_last_6mths', 'verification_status',
    'num_op_rev_tl', 'total_rev_hi_lim', 'sub_grade', 'num_bc_sats',
    'num_tl_op_past_12m', 'annual_inc', 'loan_status'
]

df_raw = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(CSV_PATH)
          .select(SELECTED_FEATURES))

print(f"Filas: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")


Filas: 2,260,701
Columnas: 21


A continuación, solamente se conservarán por los préstamos con resultado *Fully Paid* o *Charged Off* de la variable respuesta **loan_status** y se expresarán de forma binaria.

In [6]:
df_filtered = (df_raw
    .filter(F.col("loan_status").isin(["Fully Paid", "Charged Off"]))
    .withColumn("default",
        F.when(F.col("loan_status") == "Charged Off", 1).otherwise(0))
    .drop("loan_status"))

print(f"Filas tras filtrar: {df_filtered.count():,}")
df_filtered.groupBy("default").count().orderBy("default").show()


Filas tras filtrar: 1,345,309
+-------+-------+
|default|  count|
+-------+-------+
|      0|1076751|
|      1| 268558|
+-------+-------+



In [7]:
df_filtered.printSchema()
df_filtered.head(3)

root
 |-- int_rate: double (nullable = true)
 |-- dti: string (nullable = true)
 |-- num_actv_rev_tl: double (nullable = true)
 |-- fico_range_high: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- acc_open_past_24mths: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- term: string (nullable = true)
 |-- mort_acc: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- revol_util: string (nullable = true)
 |-- fico_range_low: string (nullable = true)
 |-- inq_last_6mths: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- num_op_rev_tl: double (nullable = true)
 |-- total_rev_hi_lim: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- num_bc_sats: double (nullable = true)
 |-- num_tl_op_past_12m: double (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- default: integer (nullable = false)



[Row(int_rate=13.99, dti='5.91', num_actv_rev_tl=4.0, fico_range_high='679.0', grade='C', acc_open_past_24mths=4.0, funded_amnt=3600.0, term=' 36 months', mort_acc=1.0, loan_amnt=3600.0, revol_util='29.7', fico_range_low='675.0', inq_last_6mths='1.0', verification_status='Not Verified', num_op_rev_tl=4.0, total_rev_hi_lim='9300.0', sub_grade='C4', num_bc_sats=2.0, num_tl_op_past_12m=3.0, annual_inc='55000.0', default=0),
 Row(int_rate=11.99, dti='16.06', num_actv_rev_tl=5.0, fico_range_high='719.0', grade='C', acc_open_past_24mths=4.0, funded_amnt=24700.0, term=' 36 months', mort_acc=4.0, loan_amnt=24700.0, revol_util='19.2', fico_range_low='715.0', inq_last_6mths='4.0', verification_status='Not Verified', num_op_rev_tl=20.0, total_rev_hi_lim='111800.0', sub_grade='C1', num_bc_sats=13.0, num_tl_op_past_12m=2.0, annual_inc='65000.0', default=0),
 Row(int_rate=10.78, dti='10.78', num_actv_rev_tl=3.0, fico_range_high='699.0', grade='B', acc_open_past_24mths=6.0, funded_amnt=20000.0, term=

Después de este tratamiento inicial queda un total de 1.345.309 observaciones y 43 columnas.

### Imputación de valores nulos
Antes de seguir con el procesamiento, es necesario hacer la imputación de datos faltantes en estas columnas. Para eso, se dividirán las variables en numéricas y categóricas y luego se impuatarán utilizando la medida descriptiva más acorde según el caso.


In [8]:
from pyspark.sql.types import StringType, DoubleType, IntegerType, LongType

TARGET_COL = "default"

categorical_cols = [
    f.name for f in df_filtered.schema.fields
    if isinstance(f.dataType, StringType) and f.name != TARGET_COL
]

numeric_cols = [
    f.name for f in df_filtered.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType)) and f.name != TARGET_COL
]

print(f"Categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNuméricas ({len(numeric_cols)}): {numeric_cols}")


Categóricas (11): ['dti', 'fico_range_high', 'grade', 'term', 'revol_util', 'fico_range_low', 'inq_last_6mths', 'verification_status', 'total_rev_hi_lim', 'sub_grade', 'annual_inc']

Numéricas (9): ['int_rate', 'num_actv_rev_tl', 'acc_open_past_24mths', 'funded_amnt', 'mort_acc', 'loan_amnt', 'num_op_rev_tl', 'num_bc_sats', 'num_tl_op_past_12m']


A continuación validaremos la integridad de las columnas numericas, para eso se va a buscar la presencia de valores no numéricos en las columnas clasificadas como numericas a ver si encontramos alguna columna que no haya sido clasificada correctamente

In [9]:
from pyspark.sql.functions import col

string_cols = [f.name for f in df_filtered.schema.fields if f.dataType.simpleString() == 'string']

for c in string_cols:
    invalid_count = df_filtered.filter(~col(c).rlike("^[0-9.]+$")).count()
    total_count = df_filtered.count()
    
    if invalid_count > 0:
        print(f"{c}: {invalid_count}/{total_count} valores no numéricos")

dti: 227/1345309 valores no numéricos
fico_range_high: 121/1345309 valores no numéricos
grade: 1345309/1345309 valores no numéricos
term: 1345309/1345309 valores no numéricos
revol_util: 26/1345309 valores no numéricos
fico_range_low: 150/1345309 valores no numéricos
inq_last_6mths: 101/1345309 valores no numéricos
verification_status: 1345309/1345309 valores no numéricos
total_rev_hi_lim: 1/1345309 valores no numéricos
sub_grade: 1345309/1345309 valores no numéricos


Como vemos, en la clasificación anterior no se incluyó annual_inc, a pesar de estar clasificada como string como se muestra a continuación.

In [10]:
df_filtered.select("annual_inc").printSchema()

root
 |-- annual_inc: string (nullable = true)



Por tanto, se incluirá entre las variables numericas que tienen valores no numéricos

In [11]:
TRULY_CATEGORICAL = ['grade', 'term', 'verification_status', 'sub_grade']

DIRTY_NUMERIC = ['dti', 'fico_range_high', 'revol_util', 'fico_range_low', 
                 'inq_last_6mths', 'total_rev_hi_lim', 'annual_inc']

BASE_NUMERIC = ['int_rate', 'num_actv_rev_tl', 'acc_open_past_24mths', 'funded_amnt', 
                'mort_acc', 'loan_amnt', 'num_op_rev_tl', 'num_bc_sats', 'num_tl_op_past_12m']

for c in DIRTY_NUMERIC:
    df_filtered = df_filtered.withColumn(
        c,
        F.when(F.col(c).rlike("^[0-9.]+$"), F.col(c).cast("double")).otherwise(None)
    )

numeric_cols = BASE_NUMERIC + DIRTY_NUMERIC
categorical_cols = TRULY_CATEGORICAL

print(f"Numéricas  ({len(numeric_cols)}): {numeric_cols}")
print(f"Categóricas ({len(categorical_cols)}): {categorical_cols}")

Numéricas  (16): ['int_rate', 'num_actv_rev_tl', 'acc_open_past_24mths', 'funded_amnt', 'mort_acc', 'loan_amnt', 'num_op_rev_tl', 'num_bc_sats', 'num_tl_op_past_12m', 'dti', 'fico_range_high', 'revol_util', 'fico_range_low', 'inq_last_6mths', 'total_rev_hi_lim', 'annual_inc']
Categóricas (4): ['grade', 'term', 'verification_status', 'sub_grade']


In [12]:
median_vals = {}
for col_name in numeric_cols:
    median_val = df_filtered.approxQuantile(col_name, [0.5], 0.01)[0]
    median_vals[col_name] = float(median_val) if median_val is not None else 0.0

df_imputed = df_filtered.fillna(median_vals)

mode_vals = {}

for col_name in categorical_cols:
    mode_rows = (df_imputed
                 .groupBy(col_name)
                 .count()
                 .orderBy(F.desc("count"))
                 .filter(F.col(col_name).isNotNull())
                 .limit(2)
                 .collect())
    
    if mode_rows:
        mode_vals[col_name] = mode_rows[0][col_name]
   

for col_name, mode in mode_vals.items():
    df_imputed = df_imputed.fillna({col_name: mode})

print(f"Columnas categóricas imputadas: {list(mode_vals.keys())}")
print(f"\nNulos restantes:")
df_imputed.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) 
                   for c in numeric_cols + categorical_cols]).show()

Columnas categóricas imputadas: ['grade', 'term', 'verification_status', 'sub_grade']

Nulos restantes:
+--------+---------------+--------------------+-----------+--------+---------+-------------+-----------+------------------+---+---------------+----------+--------------+--------------+----------------+----------+-----+----+-------------------+---------+
|int_rate|num_actv_rev_tl|acc_open_past_24mths|funded_amnt|mort_acc|loan_amnt|num_op_rev_tl|num_bc_sats|num_tl_op_past_12m|dti|fico_range_high|revol_util|fico_range_low|inq_last_6mths|total_rev_hi_lim|annual_inc|grade|term|verification_status|sub_grade|
+--------+---------------+--------------------+-----------+--------+---------+-------------+-----------+------------------+---+---------------+----------+--------------+--------------+----------------+----------+-----+----+-------------------+---------+
|       0|              0|                   0|          0|       0|        0|            0|          0|                 0|  0|       

Como se puede ver, ya no hay variables con nulos presentes.

### StringIndexer 
El StringIndexer convierte cada variable categórica en un indice numérico según la frecuencia. Se hace necesario de incluir, pues OneHotEncoder requiere que todos los indices sean de tipo numérico.

In [13]:
from pyspark.ml.feature import StringIndexer

indexer_output_cols = [f"{c}_idx" for c in categorical_cols]

df_indexed = df_imputed

for col_name, out_name in zip(categorical_cols, indexer_output_cols):
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=out_name,
        handleInvalid="keep"
    )
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

print(f"StringIndexer aplicado a {len(categorical_cols)} variables")



StringIndexer aplicado a 4 variables


In [44]:
df_indexed.select(indexer_output_cols).show(7)

+---------+--------+-----------------------+-------------+
|grade_idx|term_idx|verification_status_idx|sub_grade_idx|
+---------+--------+-----------------------+-------------+
|      1.0|     0.0|                    2.0|          6.0|
|      1.0|     0.0|                    2.0|          0.0|
|      0.0|     1.0|                    2.0|          1.0|
|      5.0|     1.0|                    0.0|         25.0|
|      1.0|     0.0|                    0.0|          5.0|
|      0.0|     0.0|                    2.0|          7.0|
|      0.0|     0.0|                    2.0|          8.0|
+---------+--------+-----------------------+-------------+
only showing top 7 rows


Ahora, las columnas grade_idx, term_idx, verification_status_idx y sub_grade_idx muestran la versión numérica de las variables categóricas. Cada valor corresponde a un índice asignado según la frecuencia de aparición de cada categoría, 0.0 es la categoría más frecuente y 1.0 a la siguiente y así sucesivamente.

### OneHotEncoder
El OneHotEncoder ayudará a convertir los indices numericos en vectores binarios. 

In [14]:
from pyspark.ml.feature import OneHotEncoder

encoder_output_cols = [f"{c}_ohe" for c in categorical_cols]

encoder = OneHotEncoder(
    inputCols=indexer_output_cols,
    outputCols=encoder_output_cols,
)

df_encoded = encoder.fit(df_indexed).transform(df_indexed)

print(f"OneHotEncoder aplicado a {len(categorical_cols)} variables")
df_encoded.select(encoder_output_cols).show(7, truncate=50)


OneHotEncoder aplicado a 4 variables
+-------------+-------------+-----------------------+---------------+
|    grade_ohe|     term_ohe|verification_status_ohe|  sub_grade_ohe|
+-------------+-------------+-----------------------+---------------+
|(7,[1],[1.0])|(2,[0],[1.0])|          (3,[2],[1.0])| (35,[6],[1.0])|
|(7,[1],[1.0])|(2,[0],[1.0])|          (3,[2],[1.0])| (35,[0],[1.0])|
|(7,[0],[1.0])|(2,[1],[1.0])|          (3,[2],[1.0])| (35,[1],[1.0])|
|(7,[5],[1.0])|(2,[1],[1.0])|          (3,[0],[1.0])|(35,[25],[1.0])|
|(7,[1],[1.0])|(2,[0],[1.0])|          (3,[0],[1.0])| (35,[5],[1.0])|
|(7,[0],[1.0])|(2,[0],[1.0])|          (3,[2],[1.0])| (35,[7],[1.0])|
|(7,[0],[1.0])|(2,[0],[1.0])|          (3,[2],[1.0])| (35,[8],[1.0])|
+-------------+-------------+-----------------------+---------------+
only showing top 7 rows


Tras aplicar OneHotEncoder a las variables categóricas grade, term, verification_status y sub_grade, cada columna se convirtió en un vector binario que indica qué categoría está presente en cada fila. Cada vector muestra la categoría activa con un 1.0, mientras que las demás categorías se representan implícitamente como 0. 

#### Verificación de problemas de conteo
Antes de crear el vector único, verificaremos si alguna de las columnas numéricas tiene problemas de conteo.

In [15]:
from pyspark.sql.functions import count, when, isnan

print("Verificación de nulos e infinitos en columnas numéricas:")
df_encoded.select([
    F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c)
    for c in numeric_cols
]).show()

Verificación de nulos e infinitos en columnas numéricas:
+--------+---------------+--------------------+-----------+--------+---------+-------------+-----------+------------------+---+---------------+----------+--------------+--------------+----------------+----------+
|int_rate|num_actv_rev_tl|acc_open_past_24mths|funded_amnt|mort_acc|loan_amnt|num_op_rev_tl|num_bc_sats|num_tl_op_past_12m|dti|fico_range_high|revol_util|fico_range_low|inq_last_6mths|total_rev_hi_lim|annual_inc|
+--------+---------------+--------------------+-----------+--------+---------+-------------+-----------+------------------+---+---------------+----------+--------------+--------------+----------------+----------+
|       0|              0|                   0|          0|       0|        0|            0|          0|                 0|  0|              0|         0|             0|             0|               0|         0|
+--------+---------------+--------------------+-----------+--------+---------+-------------

Como no tenemos ni nulos ni infinitos, se puede proceder a la construcción del VectorAssembler.

### Vector Assemble

Ahora, la herramienta VectorAssembler ayudará a convertir todas las columnas en un unico vector, formato requerido para entrenar el modelo.

In [16]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=numeric_cols + encoder_output_cols,
    outputCol="features_raw",
    handleInvalid="keep"
)

df_assembled = assembler.transform(df_encoded)

print(f"VectorAssembler: {len(numeric_cols + encoder_output_cols)} inputs → 'features_raw'")
df_assembled.select("features_raw").show(3, truncate=80)


VectorAssembler: 20 inputs → 'features_raw'
+--------------------------------------------------------------------------------+
|                                                                    features_raw|
+--------------------------------------------------------------------------------+
|(63,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,23,27,34],[13.99,4.0,4.0,3600.0...|
|(63,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,23,27,28],[11.99,5.0,4.0,24700....|
|(63,[0,1,2,3,4,5,6,7,9,10,11,12,14,15,16,24,27,29],[10.78,3.0,6.0,20000.0,5.0...|
+--------------------------------------------------------------------------------+
only showing top 3 rows


In [50]:
df_assembled.select("features_raw", TARGET_COL).printSchema()

root
 |-- features_raw: vector (nullable = true)
 |-- default: integer (nullable = false)



### División estratificada

Ahora, se procederá a realizar una división estratificada separando por clases y dividiendo cada conjunto con RandomSplit, el objetivo es conservar la proporción de la clase default en todos los conjuntos.

In [17]:
SEED = 42

train_0, test_0 = df_assembled.filter(F.col(TARGET_COL) == 0).randomSplit([0.8, 0.2], seed=SEED)
train_1, test_1 = df_assembled.filter(F.col(TARGET_COL) == 1).randomSplit([0.8, 0.2], seed=SEED)

df_train = train_0.union(train_1)
df_test  = test_0.union(test_1)

print(f"Train: {df_train.count():,} filas")
print(f"Test : {df_test.count():,} filas")
print("\nDistribución en TRAIN:")
df_train.groupBy(TARGET_COL).count().orderBy(TARGET_COL).show()
print("Distribución en TEST:")
df_test.groupBy(TARGET_COL).count().orderBy(TARGET_COL).show()


Train: 1,076,987 filas
Test : 268,322 filas

Distribución en TRAIN:
+-------+------+
|default| count|
+-------+------+
|      0|861811|
|      1|215176|
+-------+------+

Distribución en TEST:
+-------+------+
|default| count|
+-------+------+
|      0|214940|
|      1| 53382|
+-------+------+



Los datos quedaron divididos en una proporción 80/20, con 1,076,987 filas para entrenamiento y 268,322 para prueba. De la misma forma, verificaremos que la proporción de clases se mantuvo consistente en ambos conjuntos, en train hay 861,811 casos de Fully Paid (80%) y 215,176 de Charged Off (20%), mientras que en test hay 214,940 (80%) y 53,382 (20%) respectivamente. Esto confirma que la división estratificada funcionó correctamente: el desbalance original del dataset (~80/20) se preservó en ambas particiones, lo que garantiza que el modelo se entrena y evalúa sobre distribuciones de clases representativas.

### Standard scaler
Ahora, que ya se tiene el vector, se procederá a estandarizar a una media 0 y desviación estándar 1. 

In [18]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_train)

df_train_ready = scaler_model.transform(df_train).select("features", TARGET_COL)
df_test_ready  = scaler_model.transform(df_test).select("features", TARGET_COL)

print(f"Train listo: {df_train_ready.count():,} filas")
print(f"Test listo : {df_test_ready.count():,} filas")
print("\nEjemplo vector escalado (primera fila):")
df_train_ready.select("features").show(1, truncate=80)


Train listo: 1,076,987 filas
Test listo : 268,322 filas

Ejemplo vector escalado (primera fila):
+--------------------------------------------------------------------------------+
|                                                                        features|
+--------------------------------------------------------------------------------+
|[-1.6606581460233343,-1.4302836211992382,-1.171227152727992,1.560726390860044...|
+--------------------------------------------------------------------------------+
only showing top 1 row


Cada feature fue centrada en 0 y escalada a desviación estándar 1, equivalente al StandardScaler de scikit-learn. Los valores negativos indican que esa observación está por debajo de la media de esa feature, y los positivos que está por encima. 

## Construcción del modelo

Se utilizará primero, RandomForestClassifier, esto con el objetivo de iniciar la búsqueda de hiperparámetros. Esta será realizada manualmente iterando sobre combinaciones, equivalente al GridSearchCV de scikit-learn. Se consultó adicionalmente como optimizar la duración de este proceso y encontró que el paralelismo interno de Spark distribuye el entrenamiento de los árboles entre los cores disponibles automáticamente cuando se configura como se presenta en el codigo.

In [19]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time
import itertools

param_grid = {
    "numTrees": [10, 50, 100],
    "maxDepth": [5, 10, 15],
    "minInstancesPerNode": [1, 5]
}

N_FOLDS = 2
seeds = [42, 84]

combinaciones = list(itertools.product(
    param_grid["numTrees"],
    param_grid["maxDepth"],
    param_grid["minInstancesPerNode"]
))

print(f"Total combinaciones a probar: {len(combinaciones)}")
for c in combinaciones:
    print(f"  numTrees={c[0]}, maxDepth={c[1]}, minInstancesPerNode={c[2]}")

Total combinaciones a probar: 18
  numTrees=10, maxDepth=5, minInstancesPerNode=1
  numTrees=10, maxDepth=5, minInstancesPerNode=5
  numTrees=10, maxDepth=10, minInstancesPerNode=1
  numTrees=10, maxDepth=10, minInstancesPerNode=5
  numTrees=10, maxDepth=15, minInstancesPerNode=1
  numTrees=10, maxDepth=15, minInstancesPerNode=5
  numTrees=50, maxDepth=5, minInstancesPerNode=1
  numTrees=50, maxDepth=5, minInstancesPerNode=5
  numTrees=50, maxDepth=10, minInstancesPerNode=1
  numTrees=50, maxDepth=10, minInstancesPerNode=5
  numTrees=50, maxDepth=15, minInstancesPerNode=1
  numTrees=50, maxDepth=15, minInstancesPerNode=5
  numTrees=100, maxDepth=5, minInstancesPerNode=1
  numTrees=100, maxDepth=5, minInstancesPerNode=5
  numTrees=100, maxDepth=10, minInstancesPerNode=1
  numTrees=100, maxDepth=10, minInstancesPerNode=5
  numTrees=100, maxDepth=15, minInstancesPerNode=1
  numTrees=100, maxDepth=15, minInstancesPerNode=5


### Cross Validation manual 

PySpark no tiene StratifiedKFold nativo, por lo tanto, se replicrá el comportamiento usando randomSplit para generar los folds manualmente. Por capacidad computacional, se dividié el entrenamiento en 2, los primeros 15 folds, y luego los ultimos 3.

In [22]:
spark.conf.set("spark.default.parallelism", "6")
spark.conf.set("spark.sql.shuffle.partitions", "6")

df_train_ready.cache()
df_train_ready.count()
print("Cache listo.")

evaluator_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COL,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

def entrenar_fold(df_train_ready, combinaciones_subset, seeds, N_FOLDS, evaluator_auc, offset=0):
    resultados_parciales = []
    TOTAL = len(combinaciones)
    tiempo_global = time.time()

    for combo_idx, (num_trees, max_depth, min_inst) in enumerate(combinaciones_subset, offset + 1):
        fold_aucs = []
        tiempo_inicio = time.time()

        print(f"\n[{combo_idx}/{TOTAL}] numTrees={num_trees}, maxDepth={max_depth}, minInst={min_inst}")

        for fold_idx, seed in enumerate(seeds):
            print(f"  fold {fold_idx+1}/{N_FOLDS}...", end=" ", flush=True)

            df_folds = df_train_ready.randomSplit([1.0 / N_FOLDS] * N_FOLDS, seed=seed)
            df_val_fold   = df_folds[fold_idx]
            df_train_fold = df_folds[0] if fold_idx != 0 else df_folds[1]
            for i in range(N_FOLDS):
                if i != fold_idx and i != (0 if fold_idx != 0 else 1):
                    df_train_fold = df_train_fold.union(df_folds[i])

            rf = RandomForestClassifier(
                labelCol=TARGET_COL,
                featuresCol="features",
                numTrees=num_trees,
                maxDepth=max_depth,
                minInstancesPerNode=min_inst,
                seed=42
            )

            modelo = rf.fit(df_train_fold)
            predicciones = modelo.transform(df_val_fold)
            auc = evaluator_auc.evaluate(predicciones)
            fold_aucs.append(auc)
            print(f"AUC={auc:.4f}")

        tiempo_total = time.time() - tiempo_inicio
        auc_medio = sum(fold_aucs) / len(fold_aucs)
        elapsed = time.time() - tiempo_global
        restante = (elapsed / (combo_idx - offset)) * (TOTAL - combo_idx)

        resultados_parciales.append({
            "numTrees": num_trees,
            "maxDepth": max_depth,
            "minInstancesPerNode": min_inst,
            "auc_medio": auc_medio,
            "fold_aucs": fold_aucs,
            "tiempo_seg": round(tiempo_total, 2)
        })

        print(f"  → AUC medio={auc_medio:.4f} | tiempo={tiempo_total:.1f}s | "
              f"completado={combo_idx}/{TOTAL} | restante≈{restante/60:.1f} min")

    return resultados_parciales

resultados_parte1 = entrenar_fold(
    df_train_ready,
    combinaciones[:15],
    seeds, N_FOLDS, evaluator_auc, offset=0
)

Cache listo.

[1/18] numTrees=10, maxDepth=5, minInst=1
  fold 1/2... AUC=0.6864
  fold 2/2... AUC=0.6841
  → AUC medio=0.6853 | tiempo=27.9s | completado=1/18 | restante≈7.9 min

[2/18] numTrees=10, maxDepth=5, minInst=5
  fold 1/2... AUC=0.6873
  fold 2/2... AUC=0.6822
  → AUC medio=0.6847 | tiempo=29.2s | completado=2/18 | restante≈7.6 min

[3/18] numTrees=10, maxDepth=10, minInst=1
  fold 1/2... AUC=0.7050
  fold 2/2... AUC=0.7034
  → AUC medio=0.7042 | tiempo=42.8s | completado=3/18 | restante≈8.3 min

[4/18] numTrees=10, maxDepth=10, minInst=5
  fold 1/2... AUC=0.7046
  fold 2/2... AUC=0.7029
  → AUC medio=0.7038 | tiempo=34.8s | completado=4/18 | restante≈7.9 min

[5/18] numTrees=10, maxDepth=15, minInst=1
  fold 1/2... AUC=0.7060
  fold 2/2... AUC=0.7051
  → AUC medio=0.7056 | tiempo=91.1s | completado=5/18 | restante≈9.8 min

[6/18] numTrees=10, maxDepth=15, minInst=5
  fold 1/2... AUC=0.7076
  fold 2/2... AUC=0.7065
  → AUC medio=0.7070 | tiempo=106.3s | completado=6/18 | res

In [23]:
spark.conf.set("spark.default.parallelism", "4")
spark.conf.set("spark.sql.shuffle.partitions", "4")

resultados_parte2 = entrenar_fold(
    df_train_ready,
    combinaciones[15:],
    seeds, N_FOLDS, evaluator_auc, offset=15
)

resultados = resultados_parte1 + resultados_parte2

print(f"\nTotal combinaciones completadas: {len(resultados)}")
for r in resultados:
    print(f"  numTrees={r['numTrees']}, maxDepth={r['maxDepth']}, "
          f"minInst={r['minInstancesPerNode']} → AUC={r['auc_medio']:.4f}")


[16/18] numTrees=100, maxDepth=10, minInst=5
  fold 1/2... AUC=0.7066
  fold 2/2... AUC=0.7058
  → AUC medio=0.7062 | tiempo=247.7s | completado=16/18 | restante≈8.3 min

[17/18] numTrees=100, maxDepth=15, minInst=1
  fold 1/2... AUC=0.7129
  fold 2/2... AUC=0.7122
  → AUC medio=0.7125 | tiempo=1210.3s | completado=17/18 | restante≈12.1 min

[18/18] numTrees=100, maxDepth=15, minInst=5
  fold 1/2... AUC=0.7130
  fold 2/2... AUC=0.7124
  → AUC medio=0.7127 | tiempo=1265.7s | completado=18/18 | restante≈0.0 min

Total combinaciones completadas: 18
  numTrees=10, maxDepth=5, minInst=1 → AUC=0.6853
  numTrees=10, maxDepth=5, minInst=5 → AUC=0.6847
  numTrees=10, maxDepth=10, minInst=1 → AUC=0.7042
  numTrees=10, maxDepth=10, minInst=5 → AUC=0.7038
  numTrees=10, maxDepth=15, minInst=1 → AUC=0.7056
  numTrees=10, maxDepth=15, minInst=5 → AUC=0.7070
  numTrees=50, maxDepth=5, minInst=1 → AUC=0.6911
  numTrees=50, maxDepth=5, minInst=5 → AUC=0.6910
  numTrees=50, maxDepth=10, minInst=1 → AUC

### Mejor combinación de hiperparámetros

Se selecciona la combinación con mayor AUC promedio a través de los folds.

In [24]:
mejor = max(resultados, key=lambda x: x["auc_medio"])

print("=" * 50)
print("Mejor combinación de hiperparámetros:")
print(f"  numTrees            : {mejor['numTrees']}")
print(f"  maxDepth            : {mejor['maxDepth']}")
print(f"  minInstancesPerNode : {mejor['minInstancesPerNode']}")
print(f"  AUC medio (CV)      : {mejor['auc_medio']:.4f}")
print(f"  AUC por fold        : {[round(a, 4) for a in mejor['fold_aucs']]}")
print(f"  Tiempo total        : {mejor['tiempo_seg']}s")

Mejor combinación de hiperparámetros:
  numTrees            : 100
  maxDepth            : 15
  minInstancesPerNode : 5
  AUC medio (CV)      : 0.7127
  AUC por fold        : [0.713, 0.7124]
  Tiempo total        : 1265.69s


### Entrenamiento final con mejores hiperparámetros

Con los mejores hiperparámetros encontrados se entrena el modelo final. Este es el modelo que se evaluará sobre el conjunto de prueba, siguiendo el mismo principio que en scikit-learn donde el modelo final se re-entrena sobre todos los datos de entrenamiento.

In [25]:
inicio_final = time.time()

rf_final = RandomForestClassifier(
    labelCol=TARGET_COL,
    featuresCol="features",
    numTrees=mejor["numTrees"],
    maxDepth=mejor["maxDepth"],
    minInstancesPerNode=mejor["minInstancesPerNode"],
    seed=42
)

modelo_final = rf_final.fit(df_train_ready)
tiempo_entrenamiento = time.time() - inicio_final

print(f"Modelo final entrenado en {tiempo_entrenamiento:.2f}s")
print(f"Número de árboles: {modelo_final.getNumTrees}")

Modelo final entrenado en 979.48s
Número de árboles: 100


### Predicciones y métricas en test

Se evalúa el modelo final sobre df_test_ready usando AUC, Accuracy y F1-score, las mismas métricas usadas en la parte de scikit-learn para permitir comparación directa.

In [26]:
inicio_pred = time.time()
predicciones_test = modelo_final.transform(df_test_ready)
tiempo_prediccion = time.time() - inicio_pred

auc_test = evaluator_auc.evaluate(predicciones_test)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL,
    predictionCol="prediction",
    metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL,
    predictionCol="prediction",
    metricName="f1"
)

accuracy_test = evaluator_acc.evaluate(predicciones_test)
f1_test       = evaluator_f1.evaluate(predicciones_test)

print(f"Tiempo de predicción : {tiempo_prediccion:.2f}s")
print(f"AUC (test)           : {auc_test:.4f}")
print(f"Accuracy (test)      : {accuracy_test:.4f}")
print(f"F1-score (test)      : {f1_test:.4f}")

Tiempo de predicción : 0.09s
AUC (test)           : 0.7143
Accuracy (test)      : 0.8041
F1-score (test)      : 0.7311


### Matriz de confusión

PySpark no tiene una función directa para la matriz de confusión como scikit-learn, por lo que se construye manualmente agrupando las predicciones por clase real y predicha.

In [27]:
from pyspark.sql.functions import col

matriz = (predicciones_test
    .groupBy(TARGET_COL, "prediction")
    .count()
    .orderBy(TARGET_COL, "prediction"))

matriz.show()

matriz_collected = matriz.collect()

TP = next((r["count"] for r in matriz_collected if r[TARGET_COL]==1 and r["prediction"]==1.0), 0)
TN = next((r["count"] for r in matriz_collected if r[TARGET_COL]==0 and r["prediction"]==0.0), 0)
FP = next((r["count"] for r in matriz_collected if r[TARGET_COL]==0 and r["prediction"]==1.0), 0)
FN = next((r["count"] for r in matriz_collected if r[TARGET_COL]==1 and r["prediction"]==0.0), 0)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_manual = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nMatriz de confusión:")
print(f"           Pred 0    Pred 1")
print(f"Real 0:    {TN:<8}  {FP:<8}")
print(f"Real 1:    {FN:<8}  {TP:<8}")
print(f"\nPrecision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 manual : {f1_manual:.4f}")

+-------+----------+------+
|default|prediction| count|
+-------+----------+------+
|      0|       0.0|213124|
|      0|       1.0|  1816|
|      1|       0.0| 50760|
|      1|       1.0|  2622|
+-------+----------+------+


Matriz de confusión:
           Pred 0    Pred 1
Real 0:    213124    1816    
Real 1:    50760     2622    

Precision : 0.5908
Recall    : 0.0491
F1 manual : 0.0907


### Resumen de tiempos y métricas finales

In [28]:
print("=" * 50)
print("Resumen de tiempos:")
print(f"  Cross-validation total : {sum(r['tiempo_seg'] for r in resultados):.1f}s")
print(f"  Entrenamiento final    : {tiempo_entrenamiento:.2f}s")
print(f"  Predicción en test     : {tiempo_prediccion:.2f}s")
print(f"\nResumen de métricas finales:")
print(f"  AUC       : {auc_test:.4f}")
print(f"  Accuracy  : {accuracy_test:.4f}")
print(f"  F1-score  : {f1_test:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")

Resumen de tiempos:
  Cross-validation total : 4928.5s
  Entrenamiento final    : 979.48s
  Predicción en test     : 0.09s

Resumen de métricas finales:
  AUC       : 0.7143
  Accuracy  : 0.8041
  F1-score  : 0.7311
  Precision : 0.5908
  Recall    : 0.0491
